In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Downloading the data

In [ ]:
from data import download_tickers_history
from data.constants import TRADING_DAYS_PER_YEAR

# set the date range for the historic data
start_date = datetime(year=2022, month=1, day=1)
end_date = datetime(year=2026, month=1, day=1)
tickers = ['NVDA', 'AAPL', 'PLTR', 'DKNG', 'CAT', 'INTC', 'AMZN', 'TSLA', 'GOOG', 'MSFT']
history = download_tickers_history(start_date, end_date, tickers)

history.head()


## Black-Litterman model justification
The Black–Litterman model is a Bayesian approach to portfolio construction that addresses the main practical problem of the Markowitz model, which is highly sensitive to outliers and errors in historical data.

#### The classical Markowitz optimizer solves the following problem:
$$\max_w \quad w^T \mu - \frac{\lambda}{2} w^T \Sigma w$$
If we estimate $\mu$ as the simple historical average over the past 3 years, a stock that happened to return $+80%$ per year may receive an extremely large weight (for example, 85%), while stable assets may receive a weight of 0%.
The Markowitz model is often called an “Error Maximizer” because it magnifies errors in expected return estimates, resulting in a highly aggressive and unstable portfolio that requires frequent and costly rebalancing.

Black and Litterman proposed reversing the process:

* **Step 1**: Prior Distribution - Market Equilibrium ($\Pi$)

    Instead of estimating expected returns from historical data, the model relies on the CAPM theorem: the market is already in equilibrium, and the market portfolio $w_{\text{mkt}}$ (weighted by market capitalization) is, by definition, optimal.

    Now we are going to perform a Reverse Optimization:
    $$\Pi = \lambda \Sigma w_{\text{mkt}},$$
    where:
    
    $w_{\text{mkt}}$ is the vector of companies' weights based on market capitalization (for example, $N$ stocks normalized so that $\sum w_i = 1$);
    
    $\Sigma$ is the covariance matrix (preferably estimated using Ledoit–Wolf shrinkage);
    
    $\lambda = \frac{E(R_{\text{mkt}}) - R_f}{\sigma_{\text{mkt}}^2}$ is the market's risk aversion coefficient (typically between 2.5 and 3.5).

    The vector $\Pi$ represents the implied (equilibrium) returns embedded in current market prices.
* **Step 2**: Investor Views - $P, Q, \Omega$
    
    If an investor has no personal views (i.e., they do not believe they have an informational advantage over the market), they simply invest according to $\Pi$ and obtain the market portfolio $w_{\text{mkt}}$.

    However, if we, as a quantitative researchers, have signals, for example, GARCH predicts lower volatility for NVDA, or the ML model identifies an asset as undervalued - we can formulate views.

    Views are described by three matrices:
    1. View vector $Q$ ($K \times 1$):
        $K$ is the number of views.
    2. Projection matrix $P$ ($K \times N$):
        Maps each view to the relevant assets.
    3. View uncertainty matrix $\Omega$ ($K \times K$):
        A diagonal matrix where the diagonal elements represent the variance (uncertainty) of each view, $\omega_k$.

    where, $\Omega = \text{diag}\left( P (\tau \Sigma) P^T \right)$, and $\tau \in [0.025, 0.05]$ - a scalar representing the degree of uncertainty in the equilibrium vector $\Pi$.

    For example, with the standard $\tau = 0.05$, the model gives 95% weight to the market equilibrium and only 5% to your hypothesis. It deliberately keeps the portfolio close to the market.

* **Step 3** Black–Litterman Posterior Vector ($E[R]_{\text{BL}}$)

    The Bayesian combination of the prior ($\Pi$) and the new information ($Q$) is given by the following analytical formula:
    $$E[R]_{\text{BL}} = \left[ (\tau \Sigma)^{-1} + P^T \Omega^{-1} P \right]^{-1} \left[ (\tau \Sigma)^{-1} \Pi + P^T \Omega^{-1} Q \right]$$
    And the adjusted covariance matrix:
    $$\Sigma_{\text{BL}} = \Sigma + \left[ (\tau \Sigma)^{-1} + P^T \Omega^{-1} P \right]^{-1}$$

In [ ]:
from data.processors import log_returns

log_ret = log_returns(history)
yr_cov = np.array(log_ret.cov() * TRADING_DAYS_PER_YEAR)  # type: ignore
num_assets = yr_cov.shape[0]
weights = np.ones(num_assets) / num_assets;
expected_returns = np.array(log_ret.mean() * TRADING_DAYS_PER_YEAR)

#### Manual calculation

In [ ]:
from src.portfolio import black_litterman, get_risk_free_rate

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date, 'BACKTEST')

# 'AAPL' will be greater than 'MSFT' by 4% annually
Q = np.array([[.04]]) # vector of views values
assets = history.columns.get_level_values(0).unique()
P = np.array([[1 if t == 'AAPL' else -1 if t == 'MSFT' else 0 for t in assets]])
(bl_returns, bl_cov_matr) = black_litterman(expected_returns, yr_cov, risk_free_rate, Q, P)

print("Black-Litterman expected returns vector:")
print(bl_returns)
print("-------")
print("Black-Litterman adjusted covariance matrix")
print(bl_cov_matr)


## Data visualization

Find the Sharpe ratio

In [ ]:
from src.portfolio import get_risk_free_rate, find_max_sharpe

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

# Investor's views
# 'AAPL' will be greater than 'MSFT' by 4% annually
Q = np.array([[.04]]) # vector of views values
assets = history.columns.get_level_values(0).unique()
P = np.array([[1 if t == 'AAPL' else -1 if t == 'MSFT' else 0 for t in assets]])

(max_sharpe, stocks_w) = find_max_sharpe(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='LEDOIT_WOLF',
    returns_model='BLACK_LITTERMAN',
    views=Q,
    views_transition=P)

exact_max_ret = max_sharpe.tangency_return
exact_max_vol = max_sharpe.tangency_vol
exact_max_sharpe = max_sharpe.max_sharpe

print(f"Exact Max Sharpe Ratio: {max_sharpe.max_sharpe:.4f}")
print(f"Exact Tangency Return: {exact_max_ret:.2%}")
print(f"Exact Tangency Volatility: {exact_max_vol:.2%}")

print("Exact optimum stocks distribution:")
stocks_w


Sharpe with no investor's hypothesis

In [ ]:
from src.portfolio import get_risk_free_rate, find_max_sharpe

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

(max_no_views_sharpe, no_views_stocks_w) = find_max_sharpe(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='LEDOIT_WOLF',
    returns_model='BLACK_LITTERMAN')

exact_no_views_max_ret = max_no_views_sharpe.tangency_return
exact_no_views_max_vol = max_no_views_sharpe.tangency_vol
exact_no_views_max_sharpe = max_no_views_sharpe.max_sharpe

print(f"Exact Max Sharpe Ratio: {exact_no_views_max_sharpe:.4f}")
print(f"Exact Tangency Return: {exact_no_views_max_ret:.2%}")
print(f"Exact Tangency Volatility: {exact_no_views_max_vol:.2%}")

print("Exact optimum stocks distribution:")
no_views_stocks_w


Find Sortino ratio

In [ ]:
from src.portfolio import get_risk_free_rate, find_max_sortino

risk_free_rate = get_risk_free_rate('T_BILLS', start_date, end_date)

# Investor's views
# 'AAPL' will be greater than 'MSFT' by 4% annually
Q = np.array([[.04]]) # vector of views values
assets = history.columns.get_level_values(0).unique()
P = np.array([[1 if t == 'AAPL' else -1 if t == 'MSFT' else 0 for t in assets]])

(max_sortino, stocks_w) = find_max_sortino(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='LEDOIT_WOLF',
    returns_model='BLACK_LITTERMAN',
    views=Q,
    views_transition=P)

max_sortino_ret = max_sortino.tangency_return
max_sortino_vol = max_sortino.tangency_vol
max_sortino_sortino = max_sortino.max_sortino

print(f"Exact Max Sortino Ratio: {max_sortino.max_sortino:.4f}")
print(f"Exact Sortino Return: {max_sortino_ret:.2%}")
print(f"Exact Sortino Volatility: {max_sortino_vol:.2%}")

print("Exact optimum stocks distribution:")
stocks_w


Sortino with no investor's hypothesis

In [ ]:
(max_no_views_sortino, no_views_stocks_w) = find_max_sortino(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='LEDOIT_WOLF',
    returns_model='BLACK_LITTERMAN')

max_no_views_sortino_ret = max_no_views_sortino.tangency_return
max_no_views_sortino_vol = max_no_views_sortino.tangency_vol
max_no_views_sortino_sortino = max_no_views_sortino.max_sortino

print(f"Exact Max Sortino Ratio: {max_no_views_sortino.max_sortino:.4f}")
print(f"Exact Sortino Return: {max_no_views_sortino_ret:.2%}")
print(f"Exact Sortino Volatility: {max_no_views_sortino_vol:.2%}")

print("Exact optimum stocks distribution:")
no_views_stocks_w

Equal weights return

In [ ]:
eq_weights_ret = weights @ expected_returns
eq_weights_vol = float(np.sqrt(weights.T @ yr_cov @ weights))

print(f"Equal weights return: {eq_weights_ret:.2%}")
print(f"Sortino return - Equal weights return: {(max_sortino_ret - eq_weights_ret):.2%}")
print(f"Sortino risk - Equal weights risk: {(max_sortino_vol - eq_weights_vol):.2%}")


Optimize portfolio for the Efficient Frontier visualization

In [ ]:
from src.portfolio import optimize_portfolio

optimum_df = optimize_portfolio(
    tickers_df=history,
    rf_base='T_BILLS',
    cov_model='LEDOIT_WOLF',
    returns_model='BLACK_LITTERMAN',
    views=Q,
    views_transition=P)
optimum_df


Visualize the data

In [ ]:
from data.processors import log_returns

fig, ax = plt.subplots(figsize=(12, 6))

vol = optimum_df['vol']
sharpe = optimum_df['sharpe']
returns = optimum_df['return']
max_vol = vol.max()
max_sharpe = sharpe.max()
max_sharpe_ind = sharpe.idxmax()
log_ret = log_returns(history)
expected_returns = log_ret.mean() * TRADING_DAYS_PER_YEAR

# capital distribution line
cal_x = np.linspace(0, max_vol, 100)
cal_y = max_sharpe * cal_x + risk_free_rate

ax.plot(
    cal_x,
    cal_y,
    color='darkblue',
    alpha=0.7,
    linewidth=2, 
)
ax.spines['left'].set_position('zero')
x_label = max_vol * 0.85
y_label = max_sharpe * x_label + risk_free_rate

# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Capital Market Line',
    color='darkblue',
    fontweight='bold'
)

# mark risk free rate point
plt.plot(0, risk_free_rate, marker="o", markersize=8, markeredgecolor="red", markerfacecolor="yellow")
plt.annotate(
    'Risk free rate',
    xy=(0, risk_free_rate),
    xytext=(risk_free_rate + 0.01, risk_free_rate + 0.05),
    arrowprops=dict(facecolor='black', width=0.5, headwidth=3, headlength=4, shrink=0.05)
)

# effective frontier
ef_x = vol
ef_y = returns
ax.plot(
    ef_x,
    ef_y,
    color='darkgreen',
    alpha=0.7,
    linewidth=2,
)

# equal weights portfolio
x0 = np.ones(num_assets) / num_assets
def_x = float(np.sqrt(x0.T @ yr_cov @ x0))
def_y = np.dot(x0, expected_returns)
ax.scatter(
    def_x,
    def_y,
    color='darkred',
    alpha=0.7,
)
plt.text(def_x + 0.001, def_y - 0.001, 'Equal weights portfolio')

# S&P 500 portfolio
sp500_df = download_tickers_history(start_date, end_date, ['^GSPC'])
sp500_log_ret = log_returns(sp500_df)

sp500_x = sp500_log_ret.std().iloc[0] * np.sqrt(TRADING_DAYS_PER_YEAR)
sp500_y = sp500_log_ret.mean().iloc[0] * TRADING_DAYS_PER_YEAR
ax.scatter(
    sp500_x,
    sp500_y,
    color='red',
    alpha=0.7,
)
plt.text(sp500_x + 0.001, sp500_y + 0.01, 'S&P 500 portfolio')

x_label = max_vol * 0.85
y_label = returns.max() * 0.85
# Add text on the plot
ax.text(
    x_label,
    y_label + 0.02,
    'Efficient frontier',
    color='darkgreen',
    fontweight='bold'
)

# mark the tangency portfolio point
plt.plot(exact_max_vol, exact_max_ret, marker="*", markersize=8, markerfacecolor="red")
plt.annotate(
    'Tangency portfolio',
    xy=(exact_max_vol, exact_max_ret),
    xytext=(exact_max_vol + 0.01, exact_max_ret - 0.05),
    arrowprops=dict(facecolor='black', width=1, headwidth=3, headlength=4, shrink=0.02)
)

# mark the max Sortino point
plt.plot(max_sortino_vol, max_sortino_ret, marker="*", markersize=8, markerfacecolor="lightgreen")
plt.annotate(
    'Max Sortino',
    xy=(max_sortino_vol, max_sortino_ret),
    xytext=(max_sortino_vol + 0.01, max_sortino_ret - 0.05),
    arrowprops=dict(facecolor='black', width=1, headwidth=3, headlength=4, shrink=0.02)
)

# mark the tangency portfolio point with no trader's views
plt.plot(exact_no_views_max_vol, exact_no_views_max_ret, marker="*", markersize=8, markerfacecolor="orange")
plt.annotate(
    'Tangency portfolio (no views)',
    xy=(exact_no_views_max_vol, exact_no_views_max_ret),
    xytext=(exact_no_views_max_vol + 0.01, exact_no_views_max_ret - 0.03),
    arrowprops=dict(facecolor='black', width=1, headwidth=3, headlength=4, shrink=0.02)
)

# mark the max Sortino point with no trader's views
plt.plot(max_no_views_sortino_vol, max_no_views_sortino_ret, marker="*", markersize=8, markerfacecolor="darkgreen")
plt.annotate(
    'Max Sortino (no views)',
    xy=(max_no_views_sortino_vol, max_no_views_sortino_ret),
    xytext=(max_no_views_sortino_vol + 0.01, max_no_views_sortino_ret + 0.02),
    arrowprops=dict(facecolor='black', width=1, headwidth=5, headlength=4, shrink=0.02)
)

plt.title('Efficient Frontier')
plt.xlabel('Portfolio deviation', fontsize=12)
plt.ylabel('Expected return', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
